# Preprocessing

We predict 4 exercises to start with: **pushup, pullup, squat, jumping_jacks**. These have the most usable data once the up/down labels are merged, and they all come from `exercise_recognition_landmarks.csv` and `imagePoses.csv`. Both of those use the same MediaPipe 33-landmark skeleton.

`calisthenics_oahs_keypoints.csv` and `yoga82_handstand_keypoints.csv` only contain handstands (and use a different skeleton / scale), so they are not used yet. `Exercises.csv` has no pose data.

Output: `processed_data/poses.csv` with columns `source, label, phase, x_<landmark>, y_<landmark>` for all 33 landmarks.

In [1]:
import os
import re

import numpy as np
import pandas as pd

Load datasets:

In [2]:
recognition = pd.read_csv("../raw_data/exercise_recognition_landmarks.csv")
image_poses = pd.read_csv("../raw_data/imagePoses.csv")

print(recognition.shape, image_poses.shape)

(1372, 101) (3501, 6)


Clean `imagePoses`: drop rows where MediaPipe found no pose, drop duplicate images (same image saved under different names), and drop the base64 image column, which is most of the file size.

In [3]:
before = len(image_poses)
image_poses = image_poses.dropna(subset=["poseLandmarks"])
image_poses = image_poses.drop_duplicates(subset="base64encoded")
image_poses = image_poses.drop(columns=["base64encoded"])
print(f"imagePoses: {before} -> {len(image_poses)} rows")

imagePoses: 3501 -> 2570 rows


Normalize exercise labels: merge the up/down variants into one label and keep the phase (`up` / `down`) as a separate column. Only `exercise_recognition` has phases.

In [4]:
LABEL_MAP = {
    "pushups_up": "pushup", "pushups_down": "pushup", "pushup": "pushup",
    "pullups_up": "pullup", "pullups_down": "pullup", "pullup": "pullup",
    "squats_up": "squat", "squats_down": "squat", "squat": "squat",
    "jumping_jacks_up": "jumping_jacks", "jumping_jacks_down": "jumping_jacks",
}

recognition = recognition.assign(
    label=recognition["pose"].map(LABEL_MAP),
    phase=recognition["pose"].str.rsplit("_", n=1).str[1],
)
image_poses["label"] = image_poses["excercise"].map(LABEL_MAP)
image_poses["phase"] = None

recognition = recognition.dropna(subset=["label"])
image_poses = image_poses.dropna(subset=["label"])
print("exercise_recognition:", recognition["label"].value_counts().to_dict())
print("imagePoses:          ", image_poses["label"].value_counts().to_dict())

exercise_recognition: {'jumping_jacks': 370, 'pullup': 289, 'squat': 266, 'pushup': 246}
imagePoses:           {'squat': 324, 'pullup': 314, 'pushup': 309}


C:\Users\Justus\AppData\Local\Temp\ipykernel_35760\127701951.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  recognition = recognition.assign(


Normalize pose data into a standard set of columns. Both datasets use MediaPipe's 33 landmarks in this order:

- `exercise_recognition` names three hand points differently (`left_pinky_1`, `left_index_1`, `left_thumb_2`), so those are renamed.
- `imagePoses` stores all landmarks as one string per row (`'x: 0.79,y: 0.27,z: -0.73,visibility: 0.99,'`), so it is parsed. Its x and y are fractions of the image width and height. Multiplying by the image size converts them to pixels, otherwise the body proportions are stretched for non-square images.

Only x and y are kept. The z (depth) estimates use different scales in the two datasets and are unreliable.

In [5]:
LANDMARKS = [
    "nose", "left_eye_inner", "left_eye", "left_eye_outer", "right_eye_inner", "right_eye",
    "right_eye_outer", "left_ear", "right_ear", "mouth_left", "mouth_right",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow", "left_wrist", "right_wrist",
    "left_pinky", "right_pinky", "left_index", "right_index", "left_thumb", "right_thumb",
    "left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle",
    "left_heel", "right_heel", "left_foot_index", "right_foot_index",
]
XY_COLUMNS = [f"{axis}_{name}" for name in LANDMARKS for axis in "xy"]

recognition.columns = [
    re.sub(r"_(pinky|index)_1$|_(thumb)_2$", lambda m: "_" + (m.group(1) or m.group(2)), c)
    for c in recognition.columns
]
assert set(XY_COLUMNS) <= set(recognition.columns)

COORD_PATTERN = re.compile(r"x: ([-+\d.eE]+),y: ([-+\d.eE]+)")


def parse_landmarks(text):
    return [float(v) for pair in COORD_PATTERN.findall(text) for v in pair]


coords = np.array(image_poses["poseLandmarks"].map(parse_landmarks).tolist())
assert coords.shape[1] == 2 * len(LANDMARKS)
coords[:, 0::2] *= image_poses["imageWidth"].to_numpy()[:, None]
coords[:, 1::2] *= image_poses["imageHeight"].to_numpy()[:, None]
image_xy = pd.DataFrame(coords, columns=XY_COLUMNS, index=image_poses.index)

poses = pd.concat([
    recognition[["label", "phase"] + XY_COLUMNS].assign(source="exercise_recognition"),
    pd.concat([image_poses[["label", "phase"]], image_xy], axis=1).assign(source="imagePoses"),
], ignore_index=True)
poses = poses[["source", "label", "phase"] + XY_COLUMNS]
poses.head()

,source,label,phase,x_nose,y_nose,x_left_eye_inner,y_left_eye_inner,x_left_eye,y_left_eye,x_left_eye_outer,...,x_right_ankle,y_right_ankle,x_left_heel,y_left_heel,x_right_heel,y_right_heel,x_left_foot_index,y_left_foot_index,x_right_foot_index,y_right_foot_index
0,exercise_recognition,jumping_jacks,down,-5.889507,-57.637520,-4.656085,-62.832863,-3.302626,-63.386856,-2.032406,...,-1.359286,34.199524,3.615098,27.651157,-0.842025,35.037060,5.842190,45.971020,0.092779,45.842150
1,exercise_recognition,jumping_jacks,down,-4.255504,-62.935925,-2.977403,-67.035990,-2.215265,-67.198250,-1.494903,...,-3.125374,51.500397,4.149046,56.273407,-1.298891,54.733307,3.980098,65.370830,-5.090634,65.641780
2,exercise_recognition,jumping_jacks,down,-2.878917,-61.709988,-1.619050,-65.693750,-0.785822,-65.814340,-0.019743,...,-3.859388,48.021600,3.412772,53.354908,-2.119770,51.265694,1.994894,62.725025,-4.452602,62.494457
3,exercise_recognition,jumping_jacks,down,-4.242575,-60.371220,-3.118133,-64.416000,-2.369744,-64.603290,-1.753780,...,-2.849652,57.269690,2.699142,59.994877,-1.485475,59.729427,1.950102,68.187256,-4.573338,68.144350
4,exercise_recognition,jumping_jacks,down,-0.805543,-56.178570,-0.055174,-58.501305,0.456936,-58.473960,0.969290,...,-2.654587,50.472115,1.911180,51.426052,-2.455719,52.861732,2.699764,57.254112,-2.288348,57.803005


Normalize coordinates so every pose has the same position and scale, no matter how far the person is from the camera or where they are in the frame:

1. Subtract the mid-hip point (average of both hips), so the hips are at (0, 0).
2. Divide by the torso length (distance from mid-hip to mid-shoulder), so the torso has length 1.
3. Mirror horizontally so the shoulders are always on the right of the hips. Otherwise a push-up facing left and one facing right look like opposite poses.
4. Drop outliers with any point more than `MAX_DISTANCE` torso lengths from the hips. These are mostly bad detections or head-on photos where the torso looks very short, which blows up every coordinate.

`exercise_recognition` was already hip-centered but used a different scale (torso length about 40), and `imagePoses` was in raw image pixels, so this puts both on the same scale. The y axis still points down (image convention).

In [6]:
MAX_DISTANCE = 5


def normalize_poses(df):
    idx = {name: i for i, name in enumerate(LANDMARKS)}
    x = df[[f"x_{n}" for n in LANDMARKS]].to_numpy()
    y = df[[f"y_{n}" for n in LANDMARKS]].to_numpy()
    hip_x = (x[:, idx["left_hip"]] + x[:, idx["right_hip"]]) / 2
    hip_y = (y[:, idx["left_hip"]] + y[:, idx["right_hip"]]) / 2
    shoulder_x = (x[:, idx["left_shoulder"]] + x[:, idx["right_shoulder"]]) / 2
    shoulder_y = (y[:, idx["left_shoulder"]] + y[:, idx["right_shoulder"]]) / 2
    torso = np.hypot(shoulder_x - hip_x, shoulder_y - hip_y)
    torso = np.where(torso > 1e-6, torso, np.nan)
    facing = np.where(shoulder_x >= hip_x, 1.0, -1.0)

    out = df.copy()
    for i, name in enumerate(LANDMARKS):
        out[f"x_{name}"] = facing * (x[:, i] - hip_x) / torso
        out[f"y_{name}"] = (y[:, i] - hip_y) / torso
    coords = out[XY_COLUMNS]
    keep = coords.notna().all(axis=1) & (coords.abs().max(axis=1) <= MAX_DISTANCE)
    return out[keep].reset_index(drop=True)


normalized = normalize_poses(poses)
dropped = poses["label"].value_counts() - normalized["label"].value_counts()
print(f"Dropped {len(poses) - len(normalized)} outlier rows:", dropped.to_dict())
normalized[XY_COLUMNS].describe().T.head(10)

Dropped 52 outlier rows: {'pullup': 6, 'squat': 3, 'pushup': 43, 'jumping_jacks': 0}


,count,mean,std,min,25%,50%,75%,max
x_nose,2066.0,0.334539,0.469559,-0.710781,0.016578,0.103192,0.484443,2.094166
y_nose,2066.0,-0.931200,0.770702,-2.375163,-1.363885,-1.257194,-0.933053,3.174725
x_left_eye_inner,2066.0,0.343910,0.485972,-1.180752,0.019771,0.099489,0.495350,2.187589
y_left_eye_inner,2066.0,-0.999464,0.773392,-2.442139,-1.422716,-1.313719,-1.033795,2.832454
x_left_eye,2066.0,0.343629,0.486501,-1.391396,0.019296,0.104625,0.497698,2.089972
y_left_eye,2066.0,-1.001795,0.769986,-2.379110,-1.421345,-1.312888,-1.034482,2.816675
x_left_eye_outer,2066.0,0.342477,0.487118,-1.584582,0.016194,0.106586,0.496194,2.052858
y_left_eye_outer,2066.0,-1.004461,0.766032,-2.297819,-1.420994,-1.313030,-1.046501,2.789380
x_right_eye_inner,2066.0,0.343392,0.485001,-0.757330,0.020326,0.104563,0.495985,2.431840
y_right_eye_inner,2066.0,-1.000313,0.771376,-2.511988,-1.425182,-1.314569,-1.037356,2.695730


Combine datasets and save:

In [7]:
os.makedirs("../processed_data", exist_ok=True)
normalized.round(5).to_csv("../processed_data/poses.csv", index=False)

print("Saved processed_data/poses.csv:", normalized.shape)
pd.crosstab(normalized["label"], normalized["source"], margins=True)

Saved processed_data/poses.csv: (2066, 69)


source,exercise_recognition,imagePoses,All
label,,,
jumping_jacks,370,0,370
pullup,288,309,597
pushup,235,277,512
squat,265,322,587
All,1158,908,2066
